In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
import ast
import json
import re
from statistics import mean
import sys
sys.path.append("..")  # helpers.py лежит в корне курса
from helpers import add_user_message, chat, print_results

client = Anthropic()
model = "claude-haiku-5-5"

In [ ]:
def chat_json(messages, schema):
    """Запрос с structured outputs: возвращает JSON по схеме"""
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        output_config={"format": {"type": "json_schema", "schema": schema}},
    )
    text = next(
        block.text for block in response.content if block.type == "text"
    )
    return json.loads(text)

In [ ]:
# Новое по уроку: поле format у каждой задачи
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be
used to evaluate prompts that generate Python, JSON, or Regex specifically
for AWS-related tasks. Generate an array of JSON objects, each representing
task that requires Python, JSON, or a Regex to complete.

Each object has "task" (description of the task) and "format" (one of
"python", "json", "regex").

* Focus on tasks that can be solved by writing a single Python function,
  a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    schema = {
        "type": "object",
        "properties": {
            "tasks": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "task": {"type": "string"},
                        "format": {
                            "type": "string",
                            "enum": ["python", "json", "regex"],
                        },
                    },
                    "required": ["task", "format"],
                    "additionalProperties": False,
                },
            }
        },
        "required": ["tasks"],
        "additionalProperties": False,
    }
    return chat_json(messages, schema)["tasks"]


dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)
dataset

In [ ]:
# Новое по уроку: инструкции о формате ответа
def run_prompt(test_case):
    """Подставляет задачу в prompt и возвращает ответ Claude"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    messages = []
    add_user_message(messages, prompt)
    return chat(client, model, messages)

In [ ]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(output, test_case):
    """Выбирает проверку синтаксиса по формату задачи"""
    validators = {
        "json": validate_json,
        "python": validate_python,
        "regex": validate_regex,
    }
    return validators[test_case["format"]](output)

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""
    messages = []
    add_user_message(messages, eval_prompt)
    schema = {
        "type": "object",
        "properties": {
            "strengths": {"type": "array", "items": {"type": "string"}},
            "weaknesses": {"type": "array", "items": {"type": "string"}},
            "reasoning": {"type": "string"},
            "score": {"type": "integer"},
        },
        "required": ["strengths", "weaknesses", "reasoning", "score"],
        "additionalProperties": False,
    }
    return chat_json(messages, schema)

In [ ]:
def run_test_case(test_case):
    """Вызывает run_prompt и оценивает результат двумя grader"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    syntax_score = grade_syntax(output, test_case)

    # Новое по уроку: среднее оценки модели и оценки синтаксиса
    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "reasoning": model_grade["reasoning"],
    }


def run_eval(dataset):
    """Запускает run_test_case для каждой задачи и печатает среднюю оценку"""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    return results

In [ ]:
results = run_eval(dataset)

In [ ]:
for result in results:
    print("FORMAT:", result["test_case"]["format"])
    print("MODEL SCORE:", result["model_score"])
    print("SYNTAX SCORE:", result["syntax_score"])
    print("REASONING:", result["reasoning"])
    print("-" * 80)

In [ ]:
print_results(results)